# Python 工程化思维基础复习 — 第三阶段（2-3 个月）

这一阶段的分水岭是：代码不只是“能跑”，还要能被别人运行、维护、排查和上线。

| 模块 | 目标 |
|---|---|
| 虚拟环境与依赖 | 隔离项目依赖，保证可复现 |
| Git | 追踪变更、协作、回滚 |
| 模块化 | 拆函数、拆文件、写可读代码 |
| 云平台基础 | GCS 上传下载、BigQuery 查询 |
| ETL pipeline | 跑通 Extract → Transform → Load |

---
## 1. 虚拟环境与依赖管理

### venv + pip 基础命令

```bash
python -m venv .venv
# Windows PowerShell
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install pandas requests pytest
pip freeze > requirements.txt
pip install -r requirements.txt
```

### Poetry 基础命令

```bash
poetry init
poetry add pandas requests
poetry add --group dev pytest ruff
poetry run pytest
```

原则：项目依赖必须写进文件，不要只停留在你本机环境里。

---
## 2. Git 版本控制

### 必须会的命令

```bash
git status
git diff
git add path/to/file.py
git commit -m "Add order ETL pipeline"
git log --oneline -5
git branch
git switch -c feature/order-etl
```

### 好习惯
- 提交前先看 `git diff`
- commit 只放相关改动，不混入无关格式化
- commit message 写“做了什么”，不要只写 `update`
- 不确定时先 `git status`，别急着 reset 或 checkout

---
## 3. 函数拆分、模块化、可读代码

把脚本变成工程的第一步，是把 ETL 拆成清晰边界：

- `extract`: 从文件、API、数据库读取数据
- `transform`: 清洗、类型转换、业务计算
- `load`: 写入文件、数据库、云存储
- `main`: 组织流程，不塞复杂业务逻辑

函数名、变量名、错误信息本身就是文档。

In [ ]:
from pathlib import Path
import pandas as pd

def extract_orders(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def transform_orders(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = {"order_id", "user_id", "amount", "status"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")

    result = df.copy()
    result["amount"] = result["amount"].fillna(0).astype(float)
    result = result[result["status"] == "paid"]
    return result

def load_orders(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def run_pipeline(input_path: Path, output_path: Path) -> None:
    raw = extract_orders(input_path)
    clean = transform_orders(raw)
    load_orders(clean, output_path)


---
## 4. 基础云平台：GCS 与 BigQuery

不用一开始就掌握全部云服务，先能完成最常见动作：

- GCS：上传、下载、列出对象
- BigQuery：执行 SQL、读取结果、写入 DataFrame
- 认证：本地开发用 ADC，生产环境用 service account

### 常见依赖

```bash
pip install google-cloud-storage google-cloud-bigquery pandas pyarrow
```

生产代码里不要硬编码凭证路径和项目 ID，优先从环境变量或配置读取。

In [ ]:
# Example only. Requires GCP credentials and installed libraries.

def upload_to_gcs(bucket_name: str, source_file: str, destination_blob: str) -> None:
    from google.cloud import storage

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob)
    blob.upload_from_filename(source_file)

def query_bigquery(sql: str):
    from google.cloud import bigquery

    client = bigquery.Client()
    return client.query(sql).to_dataframe()

# df = query_bigquery("SELECT 1 AS ok")
# upload_to_gcs("my-bucket", "output/orders.csv", "daily/orders.csv")


---
## 5. 跑通一个简单 ETL pipeline

先跑通最小闭环，再逐步增加测试、日志、配置、调度。

### 最小闭环
1. 从 CSV/JSON/API 读取数据
2. 校验字段和类型
3. 清洗空值、过滤无效数据
4. 生成聚合结果
5. 写到本地文件、GCS 或 BigQuery

### 上生产前再补
- 参数配置：输入路径、输出路径、日期分区
- 日志：记录行数、耗时、失败原因
- 测试：至少覆盖 transform 函数
- 幂等性：同一天重复跑不会产生重复数据
- 失败处理：明确是否重试、跳过、报警

In [ ]:
import tempfile

input_path = Path(tempfile.gettempdir()) / "orders_input.csv"
output_path = Path(tempfile.gettempdir()) / "orders_paid.csv"

sample = pd.DataFrame({
    "order_id": [1, 2, 3],
    "user_id": [101, 102, 101],
    "amount": [100, None, 250],
    "status": ["paid", "failed", "paid"],
})
sample.to_csv(input_path, index=False)

run_pipeline(input_path, output_path)
print(pd.read_csv(output_path))

---
## 阶段验收

完成第三阶段后，你应该能做到：

1. 新项目会创建虚拟环境并固定依赖
2. 会用 Git 查看、提交、隔离自己的改动
3. 能把一个长脚本拆成 extract/transform/load/main
4. 能写一个最小可测试的 transform 函数
5. 能把结果上传到 GCS 或通过 BigQuery SQL 查询数据
6. 能跑通一个简单 ETL，并知道上线前还缺什么

练习项目：做一个每日订单汇总 pipeline：读取订单 CSV，过滤成功订单，按日期和用户聚合金额，输出 Parquet，并为 transform 写单元测试。